In [1]:
# Імпорт необхідних бібліотек

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Завантаження таблиці

df = pd.read_csv("C:\\Users\\Acer\\Downloads\\Google Ads Dataset.csv", sep=';')

# Початковий аналіз
print(df.shape)
print(df.info())
print(df.describe())
print(df.head())
print(f'Кількість дублікатів: {df.duplicated().sum()}')

(2600, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2600 entries, 0 to 2599
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Ad_ID            2600 non-null   object 
 1   Campaign_Name    2600 non-null   object 
 2   Clicks           2488 non-null   float64
 3   Impressions      2546 non-null   float64
 4   Cost             2503 non-null   object 
 5   Leads            2552 non-null   float64
 6   Conversions      2526 non-null   float64
 7   Conversion Rate  1974 non-null   float64
 8   Sale_Amount      2461 non-null   object 
 9   Ad_Date          2600 non-null   object 
 10  Location         2600 non-null   object 
 11  Device           2600 non-null   object 
 12  Keyword          2600 non-null   object 
dtypes: float64(5), object(8)
memory usage: 264.2+ KB
None
            Clicks  Impressions        Leads  Conversions  Conversion Rate
count  2488.000000  2546.000000  2552.000000  2526.000000 

Цей аналіз показує:  
  
1. Є відсутні значення, зокрема у всіх числових стовпцях.  
2. Неправильні типи даних у деяких змінних.  
3. Категоріальні стовпці містять некоректні записи та непослідовне форматування (наприклад, DataAnalyticsCourse, hyderabad/HYDERABAD).  
4. Дати також мають непослідовне форматування.  
5. У стовпцях 'Cost' та 'Sale Amount' присутні символи, через що типи даних неправильні.  
6. Стовпець 'Sale_Amount' можна перейменувати на 'Revenue', а 'Ad_Date' — просто на 'Date' для зручнішого використання.  

In [3]:
# 1.Перейменування стовпців

df = df.rename(columns={
    'Sale_Amount': 'Revenue',
    'Ad_Date': 'Date'
})

In [4]:
# 2. Коригування неправильних типів даних у Cost, Sale Amount та Ad Date

df['Cost'] = df['Cost'].str.replace('$', '').str.replace(',', '').astype(float)
df['Revenue'] = df['Revenue'].str.replace('$', '').str.replace(',', '').astype(float)
df['Date'] = df['Date'].str.replace('/', '-')
df['Date'] = pd.to_datetime(df['Date'], format='mixed', dayfirst=True)

# Створення додаткових змінних для подальшого аналізу

df['Month'] = df['Date'].dt.month_name()
df['Weekday'] = df['Date'].dt.day_name()

In [5]:
# 3. Аналіз відсутніх значень

nulls = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Percentage': (df.isnull().sum()/len(df))*100
})
print(nulls)

                 Missing Values  Percentage
Ad_ID                         0    0.000000
Campaign_Name                 0    0.000000
Clicks                      112    4.307692
Impressions                  54    2.076923
Cost                         97    3.730769
Leads                        48    1.846154
Conversions                  74    2.846154
Conversion Rate             626   24.076923
Revenue                     139    5.346154
Date                          0    0.000000
Location                      0    0.000000
Device                        0    0.000000
Keyword                       0    0.000000
Month                         0    0.000000
Weekday                       0    0.000000


Результати показали значну кількість пропусків у показнику Conversion Rate — 24%, але цей показник є похідним і може бути перерахований. Тому спочатку необхідно перевірити, яка саме формула використовується для Conversion Rate: Conversions/Clicks чи Leads/Clicks.

In [6]:
# Створення таблиці без null-значень у стовпцях Clicks, Conversions і Conversion Rate

df_nonulls = df.dropna(subset=['Clicks', 'Conversions', 'Conversion Rate']).copy()

# Обчислення Conversion Rate як Conversions/Clicks та порівняння з оригінальним Conversion Rate

df_nonulls['Calculated CR'] = df_nonulls['Conversions']/df_nonulls['Clicks']*100
df_nonulls['Conversion Rate'] = df_nonulls['Conversion Rate']*100
df_nonulls['CR Difference'] = abs(df_nonulls['Conversion Rate'] - df_nonulls['Calculated CR'])
df_nonulls['Calculated CR'] = df_nonulls['Calculated CR'].round(2)

print(df_nonulls[['Clicks', 'Conversions', 'Conversion Rate', 'Calculated CR', 'CR Difference']].head(15))

    Clicks  Conversions  Conversion Rate  Calculated CR  CR Difference
0    104.0          7.0              5.8           6.73       0.930769
1    173.0          8.0              4.6           4.62       0.024277
6    116.0          5.0              4.3           4.31       0.010345
7    184.0          3.0              1.6           1.63       0.030435
8    113.0          4.0              5.8           3.54       2.260177
9    166.0          9.0              5.4           5.42       0.021687
10   101.0          6.0              5.9           5.94       0.040594
11   101.0          5.0              5.0           4.95       0.049505
12   125.0          3.0              2.4           2.40       0.000000
13   196.0          7.0              3.8           3.57       0.228571
14   181.0          9.0              5.0           4.97       0.027624
15   102.0          8.0              7.8           7.84       0.043137
16   193.0          9.0              4.7           4.66       0.036788
17   1

Результати показали, що більшість значень відповідають формулі Conversions/Clicks, однак все ще є помітні відхилення, наприклад 2.3, 0.9 або 0.2. Хоча значення до 0.5 можуть пояснюватися звичайним округленням, усе, що перевищує цей поріг, уже виглядає як помітна різниця. Відхилення понад 1 є ще більш суттєвими та можуть свідчити про потенційні помилки під час введення даних. Важливо порахувати, скільки таких значень присутні в наборі даних.

In [7]:
# Перевірка кількості відхилень, більших за 0.5 та 1

noticeable_diff = df_nonulls[df_nonulls['CR Difference'] > 0.5]
significant_diff = df_nonulls[df_nonulls['CR Difference'] > 1]
print(f'Кількість рядків з різницею більше 0.5: {len(noticeable_diff)} з {len(df_nonulls)}, тобто {len(noticeable_diff)/len(df_nonulls)*100:.2f}%')
print(f'Кількість рядків з різницею більше 1: {len(significant_diff)} з {len(df_nonulls)}, тобто {len(significant_diff)/len(df_nonulls)*100:.2f}%')

Кількість рядків з різницею більше 0.5: 420 з 1944, тобто 21.60%
Кількість рядків з різницею більше 1: 331 з 1944, тобто 17.03%


Аналіз виявив 21.5% та 17% для розбіжностей більше 0.5 та 1 відповідно. Це доволі високі показники, які свідчать про те, що надійніше буде перерахувати весь стовпець Conversion Rate. Але для цього спершу потрібно вирішити проблему з null-значеннями в інших стовпцях, а також проблему з некоректними записами в категоріальних стовпцях щоб використовувати їх в подальшому аналізі. 

In [8]:
# Перевірка змінних, у яких були помічені некоректні значення

cat_cols = ['Campaign_Name', 'Location', 'Device', 'Keyword']

for col in cat_cols:
    print(df[col].unique())

['DataAnalyticsCourseBasics' 'Data Anlytics Corse Vizulizations'
 'Data Analytcis Course Advansed' 'Data Analytics Corse ML']
['hyderabad' 'HYDERABAD' 'Hyderbad' 'hydrebad']
['desktop' 'mobile' 'Desktop' 'tablet' 'MOBILE' 'TABLET' 'Tablet' 'Mobile'
 'DESKTOP']
['learn data analytics' 'data analytics course' 'data analitics online'
 'data anaytics training' 'online data analytic' 'analytics for data']


In [9]:
# Виправлення невідповідних значень

# Приведення даних до єдиного формату
for col in ['Campaign_Name', 'Location', 'Device', 'Keyword']:
    df[col] = df[col].str.strip().str.capitalize()
    
# Усунення помилок в значеннях
df['Campaign_Name'] = df['Campaign_Name'].replace({
    'Dataanalyticscoursebasics': 'Data Analytics Course: Basics',
    'Data anlytics corse vizulizations': 'Data Analytics Course: Visualizations',
    'Data analytcis course advansed': 'Data Analytics Course: Advanced',
    'Data analytics corse ml': 'Data Analytics Course: ML'
})

df['Location'] = df['Location'].replace({
    'Hyderbad': 'Hyderabad',
    'Hydrebad': 'Hyderabad'
})

df['Keyword'] = df['Keyword'].replace({
    'Data analitics online': 'Data analytics online',
    'Data anaytics training': 'Data analytics training',
    'Online data analytic': 'Online data analytics'
})

for col in cat_cols:
    print(df[col].unique())

['Data Analytics Course: Basics' 'Data Analytics Course: Visualizations'
 'Data Analytics Course: Advanced' 'Data Analytics Course: ML']
['Hyderabad']
['Desktop' 'Mobile' 'Tablet']
['Learn data analytics' 'Data analytics course' 'Data analytics online'
 'Data analytics training' 'Online data analytics' 'Analytics for data']


Всі неправильно заповнені значення було виправлено.
В результаті бачимо, що є чотири рекламні кампанії для курсів з дата аналітики за їх основними напрямами. Кампанії запускаються в одному місті, перевіряємо три різні девайси і шість ключових слів.

Тепер повернемося до пропущених значень. Інші змінні з null-значеннями — це Clicks, Impressions, Leads, Conversions, Cost і Revenue. Майже всі вони мають менше 5% пропусків, однак видалення всіх або навіть частини таких рядків може призвести до значної втрати даних, тому краще обережно заповнити їх з урахуванням інших змінних. Однак я збираюсь видалити рядки, які містять більше одного пропущеного значення, щоб отримати більш надійні результати.  
Перед заповненням також перевірю чи існує зв’язок між пропущеними значеннями Clicks або Impressions та змінними Device, Weekday чи Keyword.

In [10]:
# Перевірка наявності зв’язку між пропущеними значеннями та змінними Device, Weekday і Keyword

print(df[df[['Clicks','Impressions']].isnull().any(axis=1)][['Weekday']].value_counts())
print(df[df[['Clicks','Impressions']].isnull().any(axis=1)][['Device']].value_counts())
print(df[df[['Clicks','Impressions']].isnull().any(axis=1)][['Keyword']].value_counts())

Weekday  
Saturday     34
Friday       31
Thursday     22
Monday       20
Wednesday    20
Tuesday      19
Sunday       17
Name: count, dtype: int64
Device 
Desktop    56
Tablet     56
Mobile     51
Name: count, dtype: int64
Keyword                
Data analytics training    32
Data analytics online      31
Data analytics course      29
Analytics for data         28
Learn data analytics       23
Online data analytics      20
Name: count, dtype: int64


Результати показали, що помітних відмінностей у розподілі пропущених значень немає. Їх не стає суттєво більше в якісь конкретні дні, на якомусь конкретному девайсі чи ключовому слові, тобто якоїсь очевидної причини втрати значень не спостерігається. Це свідчить про те, що вони, ймовірно, були пропущені випадково і їх можна заповнити за допомогою загального середнього значення допоміжних показників.  

In [11]:
# Заповнення втрачених значень:

# Clicks
ctr = (df['Clicks'] / df['Impressions'])
median_ctr = ctr[(df['Clicks'].notna()) & (df['Impressions'].notna()) & (df['Impressions'] > 0)].median()
df.loc[df['Clicks'].isnull() & df['Impressions'].notnull(), 'Clicks'] = (
    df['Impressions'] * median_ctr)

# Impressions
df.loc[df['Impressions'].isnull() & df['Clicks'].notnull(), 'Impressions'] = (
    df['Clicks'] / median_ctr)

# Leads
lead_rate = (df['Leads'] / df['Clicks'])
median_lead_rate = lead_rate[(df['Leads'].notna()) & (df['Clicks'] > 0)].median()
df.loc[df['Leads'].isnull(), 'Leads'] = (
    df['Clicks'] * median_lead_rate)

# Conversions
conv_rate = (df['Conversions'] / df['Clicks'])
median_conv_rate = conv_rate[(df['Conversions'].notna()) & (df['Clicks'] > 0)].median()
df.loc[df['Conversions'].isnull(), 'Conversions'] = (
    df['Clicks'] * median_conv_rate)

# Cost
mask = (df['Cost'].notna()) & (df['Clicks'] > 0)
df.loc[mask, 'CPC'] = df.loc[mask, 'Cost'] / df.loc[mask, 'Clicks']
median_cpc_per_device = df.loc[mask].groupby('Device')['CPC'].median()
mask_missing_cost = df['Cost'].isna() & (df['Clicks'] > 0)
df.loc[mask_missing_cost, 'Cost'] = df.loc[mask_missing_cost].apply(
    lambda row: row['Clicks'] * median_cpc_per_device[row['Device']], axis=1)


# Revenue 
rev_per_conv = (df['Revenue'] / df['Conversions'])
median_rev = rev_per_conv[(df['Revenue'].notna()) & (df['Conversions'] > 0)].median()
df.loc[df['Revenue'].isnull(), 'Revenue'] = (
    df['Conversions'] * median_rev)

# Conversion Rate 
df['Conversion Rate'] = df['Conversions']/df['Clicks']

# Видалення рядків з більш ніж одним втраченим значенням
df = df.drop(df[df['Clicks'].isnull() & df['Impressions'].isnull()].index)

# Округлення чисел для більшої зручності
for col in ['Clicks', 'Impressions', 'Leads', 'Conversions', 'Cost']:
    df[col] = df[col].round()

# Перевірка перших 15 рядків, щоб побачити чи немає дивних значень
print(df.head(15))

    Ad_ID                          Campaign_Name  Clicks  Impressions   Cost  \
0   A1000          Data Analytics Course: Basics   104.0       4498.0  232.0   
1   A1001          Data Analytics Course: Basics   173.0       5107.0  217.0   
2   A1002  Data Analytics Course: Visualizations    90.0       4544.0  204.0   
3   A1003        Data Analytics Course: Advanced   142.0       3185.0  238.0   
4   A1004              Data Analytics Course: ML   156.0       3361.0  196.0   
5   A1005          Data Analytics Course: Basics   195.0       3776.0  244.0   
6   A1006              Data Analytics Course: ML   116.0       4480.0  238.0   
7   A1007              Data Analytics Course: ML   184.0       5060.0  230.0   
8   A1008              Data Analytics Course: ML   113.0       5434.0  175.0   
9   A1009        Data Analytics Course: Advanced   166.0       3355.0  187.0   
10  A1010          Data Analytics Course: Basics   101.0       5399.0  237.0   
11  A1011              Data Analytics Co

In [12]:
# Перевірка кількості втрачених значень після заповнення

nulls = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Percentage': (df.isnull().sum()/len(df))*100
})
print(nulls)

                 Missing Values  Percentage
Ad_ID                         0    0.000000
Campaign_Name                 0    0.000000
Clicks                        0    0.000000
Impressions                   0    0.000000
Cost                          0    0.000000
Leads                         0    0.000000
Conversions                   0    0.000000
Conversion Rate               0    0.000000
Revenue                       0    0.000000
Date                          0    0.000000
Location                      0    0.000000
Device                        0    0.000000
Keyword                       0    0.000000
Month                         0    0.000000
Weekday                       0    0.000000
CPC                          97    3.735079


Всі втрачені значення зникли. Тепер можна перерахувати CPC та інші необхідні для аналізу метрики.

In [13]:
# Перераховуємо всі інші необхідні метрики

df['CTR'] = (df['Clicks'] / df['Impressions'] * 100).round(2)
df['CPC'] = (df['Cost'] / df['Clicks']).round(2)
df['CPL'] = (df['Cost'] / df['Leads']).round(2)
df['CPA'] = (df['Cost'] / df['Conversions']).round(2)
df['ROI'] = ((df['Revenue'] - df['Cost']) / df['Cost'] * 100).round(2)
df['AOV'] = (df['Revenue'] / df['Conversions']).round(2)
df['ROAS'] = (df['Revenue'] / df['Cost']).round(2)

In [14]:
# Швидкий поверхневий аналіз всих метрик

metrics = df[['CTR', 'CPC', 'CPL', 'CPA', 'ROI', 'AOV', 'ROAS']]
metrics.describe()

,CTR,CPC,CPL,CPA,ROI,AOV,ROAS
count,2597.000000,2597.000000,2597.000000,2597.000000,2597.000000,2597.000000,2597.000000
mean,3.180112,1.651814,11.963231,38.113242,43.712811,53.813007,1.437181
std,1.012078,0.460864,4.313749,16.283865,33.220000,24.345512,0.332257
min,1.360000,0.910000,4.300000,12.700000,-43.720000,20.400000,0.560000
25%,2.450000,1.290000,8.480000,25.440000,17.970000,36.290000,1.180000
50%,3.030000,1.550000,10.700000,32.710000,41.840000,46.710000,1.420000
75%,3.740000,1.940000,14.670000,47.000000,66.840000,66.000000,1.670000
max,6.550000,3.090000,26.500000,98.670000,186.400000,135.330000,2.860000


In [15]:
# Збереження очищеного датасету для подальшого використання

df.to_csv("C:\\Users\\Acer\\Downloads\\Google Ads Dataset(Cleaned).csv")